# Prepare a defensible training distribution

The checked-in/test-bag derivative is for smoke tests only. Use this notebook with the official ChessBench training action-value bag and a mounted 270M checkpoint. Run it once, preserve the outputs as a Kaggle Dataset, and attach that Dataset to all six training kernels.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/working/chess-slm-benchmark')
SL_REPO = Path('/kaggle/working/searchless_chess')
CHESSBENCH = Path('/kaggle/input/chessbench-train')  # change Dataset name
BAG = CHESSBENCH / 'action_value_data.bag'
assert BAG.exists(), BAG
os.chdir(REPO)
%pip install -q python-chess zstandard apache-beam grain jaxtyping

In [ ]:
# First run with MAX_RECORDS=10000 to validate the mounted bag and tokenizer.
MAX_RECORDS = 0  # set 10000 for a smoke test; 0 means the full bag
FULL_OUT = Path('/kaggle/working/chessbench-full')
FULL_OUT.mkdir(parents=True, exist_ok=True)
TRAIN_SET = FULL_OUT / 'train_set.npz'
cmd = [sys.executable, 'scripts/build_student_train_set.py', '--bag', str(BAG), '--sl-repo', str(SL_REPO), '--out', str(TRAIN_SET)]
if MAX_RECORDS:
    cmd += ['--max-records', str(MAX_RECORDS)]
subprocess.run(cmd, check=True)
print(TRAIN_SET, TRAIN_SET.stat().st_size / 2**30, 'GiB')

## Teacher labeling

Set `TEACHER_PARAMS` to the mounted 270M `params` directory. The old Orbax/JAX stack may require a dedicated Kaggle image; do not silently downgrade or change the teacher if imports fail. Record the exact error and environment.

In [ ]:
TEACHER_PARAMS = Path('/kaggle/input/searchless-270m/6400000/params')  # change
TEACHER_OUT = FULL_OUT / 'teacher_logp.npy'
assert TEACHER_PARAMS.exists(), TEACHER_PARAMS
cmd = [sys.executable, 'scripts/teacher_label.py', '--npz', str(TRAIN_SET), '--checkpoint', str(TEACHER_PARAMS), '--out', str(TEACHER_OUT), '--batch', '128', '--sl-repo', str(SL_REPO), '--dim', '1024', '--layers', '16', '--heads', '8']
subprocess.run(cmd, check=True)
print(TEACHER_OUT, TEACHER_OUT.stat().st_size / 2**30, 'GiB')

In [ ]:
import numpy as np
d = np.load(TRAIN_SET)
t = np.load(TEACHER_OUT, mmap_mode='r')
assert d['tokens'].shape[0] == t.shape[0]
assert t.shape[1] == 128
assert np.allclose(np.logaddexp.reduce(np.asarray(t[:1024]), axis=1), 0, atol=2e-3)
print('validated training rows:', len(t))